# ML Zoomcamp 2026 – Homework 6: Decision Trees and Ensemble Learning

**Autor:** Angel Uriel Monterrosas Gonzalez

**Fecha:** [02 - 11 - 2026]

**Módulo:** 06 – Decision Trees and Ensemble Learning

**Actividad:** Homework 06

**Dataset:** [car_fuel_efficiency_2026.csv](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv)

Este notebook responde a las preguntas del homework oficial disponible en:
https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06

**Objetivo:** Comparar árboles de decisión, Random Forest y XGBoost para predecir `fuel_efficiency_mpg`.

## 2. Objetivos de aprendizaje

- Entrenar un árbol de decisión y extraer la feature usada en el primer split.
- Entrenar un Random Forest y evaluar RMSE en validación.
- Explorar el efecto de `n_estimators` y `max_depth` sobre el RMSE.
- Extraer la importancia de features de un bosque aleatorio.
- Entrenar un modelo XGBoost con API nativa (DMatrix + watchlist).
- Ajustar el learning rate (`eta`) y comparar resultados.

## 3. Setup

Importamos librerías, fijamos semilla y definimos la URL del dataset 2026.

In [2]:
# 3.1 Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import xgboost as xgb

# 3.2 Versiones
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("XGBoost version:", xgb.__version__)

# 3.3 Semilla global
SEED = 1
np.random.seed(SEED)

# 3.4 URL del dataset (2026)
DATASET_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

Pandas version: 3.0.6
NumPy version: 2.5.3
XGBoost version: 3.4.1


## 4. Carga y preparación del dataset

Cargamos el CSV, rellenamos valores faltantes con 0 y hacemos el split exacto del homework.

In [3]:
# 4.1 Carga
df = pd.read_csv(DATASET_URL)
print("Shape:", df.shape)
df.head()

Shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
# 4.2 Rellenar valores faltantes con 0 (indicado en el homework)
df = df.fillna(0)
print("Valores faltantes restantes:", df.isna().sum().sum())

Valores faltantes restantes: 0


In [5]:
# 4.3 Split exacto (seed=1)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 6000, Val: 2000, Test: 2000


In [6]:
# 4.4 Separar features y target
target = "fuel_efficiency_mpg"

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values
y_full_train = df_full_train[target].values

del df_train[target]
del df_val[target]
del df_test[target]
del df_full_train[target]

print("y_train:", y_train.shape, "| y_val:", y_val.shape, "| y_test:", y_test.shape)

y_train: (6000,) | y_val: (2000,) | y_test: (2000,)


## 5. Vectorización con DictVectorizer

El homework pide `DictVectorizer(sparse=True)`. Usamos los diccionarios del dataframe y ajustamos sobre el training set.

In [7]:
# 5.1 DictVectorizer sparse=True
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(df_train.to_dict(orient="records"))
X_val = dv.transform(df_val.to_dict(orient="records"))

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Número de features tras one-hot:", len(dv.feature_names_))

X_train shape: (6000, 16)
X_val shape: (2000, 16)
Número de features tras one-hot: 16


## Pregunta 1 – Feature usada en el primer split

Entrenamos un árbol de decisión con `max_depth=1` y miramos cuál es la primera feature usada para dividir. Con `DictVectorizer`, una feature como `origin=USA` corresponde a la columna original `origin`.

In [8]:
# Q1: Decision Tree con max_depth=1
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# Obtener la feature usada en el primer split
feature_names = np.array(dv.feature_names_)
importances = dt.feature_importances_
top_idx = np.argmax(importances)
top_feature = feature_names[top_idx]

print(f"Feature usada en el primer split: {top_feature}")
print(f"Importancia: {importances[top_idx]:.4f}")

# Mapear si es one-hot (e.g., 'origin=USA' → 'origin')
original_feature = top_feature.split("=")[0]
print(f"\nColumna original: {original_feature}")

# También podemos ver el árbol
print("\nEstructura del árbol:")
print(export_text(dt, feature_names=list(feature_names)))

Feature usada en el primer split: model_year
Importancia: 1.0000

Columna original: model_year

Estructura del árbol:
|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Pregunta 2 – RMSE del Random Forest en validación

Entrenamos un Random Forest con `n_estimators=10`, `random_state=1`, `n_jobs=-1` y calculamos el RMSE sobre validación.

In [9]:
# 6.0 Función RMSE reutilizable
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [10]:
# Q2: Random Forest
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)
rmse_rf = rmse(y_val, y_pred_rf)
print(f"RMSE en validación (n_estimators=10): {rmse_rf:.4f}")

RMSE en validación (n_estimators=10): 1.8332


## Pregunta 3 – Barrido de `n_estimators`

Probamos `[10, 50, 100, 150]` con `random_state=1` y comparamos el RMSE en validación (3 decimales).

In [11]:
# Q3: Barrido de n_estimators
n_estimators_values = [10, 50, 100, 150]
results_n = {}

for n in n_estimators_values:
    rf_n = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    rf_n.fit(X_train, y_train)
    y_pred_n = rf_n.predict(X_val)
    score = rmse(y_val, y_pred_n)
    results_n[n] = round(score, 3)
    print(f"n_estimators={n}: RMSE = {score:.3f}")

best_n = min(results_n, key=results_n.get)
print(f"\nMejor n_estimators: {best_n} (RMSE={results_n[best_n]})")

n_estimators=10: RMSE = 1.833
n_estimators=50: RMSE = 1.774
n_estimators=100: RMSE = 1.770
n_estimators=150: RMSE = 1.769

Mejor n_estimators: 150 (RMSE=1.769)


## Pregunta 4 – Mejor `max_depth` por RMSE medio

Para cada `max_depth` en `[10, 15, 20, 25]` probamos todos los `n_estimators` en `[10, 50, 100, 150]`, calculamos el RMSE en validación y tomamos la media. El `max_depth` con menor RMSE medio es el ganador.

In [12]:
# Q4: Barrido de max_depth × n_estimators
max_depth_values = [10, 15, 20, 25]
results_depth = {}

for md in max_depth_values:
    scores = []
    for n in [10, 50, 100, 150]:
        rf_md = RandomForestRegressor(
            n_estimators=n,
            max_depth=md,
            random_state=1,
            n_jobs=-1
        )
        rf_md.fit(X_train, y_train)
        y_pred_md = rf_md.predict(X_val)
        scores.append(rmse(y_val, y_pred_md))
    mean_rmse = np.mean(scores)
    results_depth[md] = round(mean_rmse, 3)
    print(f"max_depth={md}: RMSE medio = {mean_rmse:.3f}")

best_md = min(results_depth, key=results_depth.get)
print(f"\nMejor max_depth: {best_md} (RMSE medio={results_depth[best_md]})")

max_depth=10: RMSE medio = 1.754
max_depth=15: RMSE medio = 1.784
max_depth=20: RMSE medio = 1.787
max_depth=25: RMSE medio = 1.786

Mejor max_depth: 10 (RMSE medio=1.754)


## Pregunta 5 – Feature más importante (Random Forest)

Entrenamos un Random Forest con `n_estimators=10, max_depth=20, random_state=1` y extraemos `feature_importances_`. Filtramos las columnas originales del listado de la pregunta.

In [13]:
# Q5: Random Forest para feature importance
rf_imp = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf_imp.fit(X_train, y_train)

importances = rf_imp.feature_importances_
feature_names = np.array(dv.feature_names_)

# Agregar importancias por columna original (maneja one-hot: 'origin=USA' → 'origin')
agg = {}
for name, imp in zip(feature_names, importances):
    original = name.split("=")[0]  # para one-hot
    agg[original] = agg.get(original, 0.0) + imp

# Filtrar a las 4 candidatas
candidates = ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]
print("Importancia por feature original (candidatas):")
for c in candidates:
    print(f"  {c}: {agg.get(c, 0.0):.4f}")

most_important = max(candidates, key=lambda c: agg.get(c, 0.0))
print(f"\nFeature más importante: {most_important}")

# (Opcional) Top-10 global
print("\nTop-10 global (columnas originales):")
for feat, imp in sorted(agg.items(), key=lambda x: -x[1])[:10]:
    print(f"  {feat}: {imp:.4f}")

Importancia por feature original (candidatas):
  vehicle_weight: 0.2033
  horsepower: 0.0783
  acceleration: 0.0550
  engine_displacement: 0.0613

Feature más importante: vehicle_weight

Top-10 global (columnas originales):
  model_year: 0.2961
  vehicle_weight: 0.2033
  fuel_type: 0.1936
  horsepower: 0.0783
  engine_displacement: 0.0613
  acceleration: 0.0550
  origin: 0.0521
  drivetrain: 0.0337
  num_doors: 0.0194
  num_cylinders: 0.0073


## Pregunta 6 – XGBoost y `eta`

Entrenamos dos modelos con la API nativa de XGBoost (DMatrix + watchlist) para 100 rondas:
- `eta=0.3`
- `eta=0.1`

Comparamos el RMSE en validación.

In [14]:
# Q6: XGBoost con DMatrix
features = list(dv.feature_names_)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

base_params = {
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

results_eta = {}
for eta in [0.3, 0.1]:
    params = {**base_params, 'eta': eta}
    watchlist = [(dtrain, 'train'), (dval, 'val')]
    model_xgb = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False,
    )
    y_pred_xgb = model_xgb.predict(dval)
    score = rmse(y_val, y_pred_xgb)
    results_eta[eta] = round(score, 4)
    print(f"eta={eta}: RMSE en validación = {score:.4f}")

best_eta = min(results_eta, key=results_eta.get)
print(f"\nMejor eta: {best_eta} (RMSE={results_eta[best_eta]})")

eta=0.3: RMSE en validación = 1.8265
eta=0.1: RMSE en validación = 1.7248

Mejor eta: 0.1 (RMSE=1.7248)


## 7. Evaluación – Resumen de resultados

Consolidamos las respuestas obtenidas para cada pregunta.

In [15]:
# 7.1 Resumen
print("=" * 55)
print("RESUMEN DE RESULTADOS – HW06")
print("=" * 55)
print(f"Q1 - Feature del primer split: {original_feature}")
print(f"Q2 - RMSE del Random Forest: {rmse_rf:.4f}")
print(f"Q3 - Mejor n_estimators: {best_n}")
print(f"Q4 - Mejor max_depth: {best_md}")
print(f"Q5 - Feature más importante: {most_important}")
print(f"Q6 - Mejor eta: {best_eta}")

RESUMEN DE RESULTADOS – HW06
Q1 - Feature del primer split: model_year
Q2 - RMSE del Random Forest: 1.8332
Q3 - Mejor n_estimators: 150
Q4 - Mejor max_depth: 10
Q5 - Feature más importante: vehicle_weight
Q6 - Mejor eta: 0.1


## 8. Interpretación de resultados

- **Q1:** El árbol con `max_depth=1` usa **[model_year]** para dividir.
- **Q2:** El Random Forest con `n_estimators=10` obtiene un RMSE de **[1.8332]**.
- **Q3:** El `n_estimators` con menor RMSE es **[150]**.
- **Q4:** El mejor `max_depth` (por RMSE medio) es **[10]**.
- **Q5:** La feature más importante es **[vehicle_weight]**.
- **Q6:** El `eta` que da menor RMSE es **[0.1]**.


## 9. Conclusiones

- **Procesamiento de datos:** Se cargó y preparó exitosamente el dataset `car_fuel_efficiency_2026.csv` con 10,000 registros y 11 columnas, rellenando los valores faltantes con 0 y aplicando una división estricta 60/20/20 (semilla `SEED=1`).
- **Árboles de Decisión:** Mediante la exploración con un árbol de profundidad `max_depth=1`, se identificó que la variable clave para realizar el primer split predictivo es **`model_year`**.
- **Random Forest:** Se comprobó que incrementar el número de estimadores mejora la estabilidad y reduce el error de validación, siendo **`n_estimators=150`** el valor óptimo evaluado. Asimismo, un `max_depth=10` arrojó el mejor rendimiento promedio al controlar el sobreajuste.
- **Importancia de Variables:** El análisis de importancia de características en el bosque aleatorio determinó que **`vehicle_weight`** (peso del vehículo) es la variable individual con mayor peso predictivo para estimar la eficiencia de combustible (`fuel_efficiency_mpg`), seguida de cerca por características temporales y mecánicas.
- **Optimización con XGBoost:** El entrenamiento mediante la API nativa de XGBoost demostró que ajustar el índice de aprendizaje a **`eta=0.1`** genera predicciones más precisas (menor RMSE en validación) en comparación con tasas de aprendizaje más altas como `0.3`.

## 10. Referencias

- Dataset: [car_fuel_efficiency_2026.csv](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv)
- Homework oficial: [ML Zoomcamp 2026 – HW06](https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06)
- Documentación RandomForestRegressor: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
- Documentación XGBoost: https://xgboost.readthedocs.io/